# Mini-TP 7 · Seguridad con SAIF — Data Lake (MinIO/S3)

**Operaciones de Aprendizaje Automático II · CEIA – FIUBA**

Este es **uno de los seis notebooks guiados** del Mini-TP 7. Cada uno asegura **una capa**
de las que construimos en el bimestre. **Elige este si tu modelo se expone por Data Lake (MinIO/S3)**
(o simplemente para practicar esta estrategia). Con completar **uno** alcanza para la entrega.

### SAIF en esta capa
**Riesgo:** **exfiltración** de datos/modelos y acceso excesivo (demasiados permisos).  
**Controles:** acceso mínimo por bucket/prefijo · versionado + *checksum* (integridad) · URLs prefirmadas con expiración.  
**Elementos SAIF:** bases sólidas · coherencia de controles.

### Qué vas a hacer
1. Ver la **versión vulnerable** y por qué es un riesgo.
2. Aplicar el **control principal** (guiado, ya resuelto como ejemplo).
3. Completar los **`# TODO`** sobre tu propio modelo.
4. Escribir una **reflexión** (amenaza mitigada ↔ elemento de SAIF).

> Corre de punta a punta con **uv**, sin servicios externos.


### Dependencias y MinIO
```bash
uv add boto3
```
Levanta MinIO con Docker (igual que en la Sesión 6):
```bash
docker run -d --name minio -p 9000:9000 -p 9001:9001 \
  -e MINIO_ROOT_USER=minio -e MINIO_ROOT_PASSWORD=minio123 \
  quay.io/minio/minio server /data --console-address ":9001"
```

In [ ]:
import os, io, hashlib, json, joblib, numpy as np
os.environ.setdefault("AWS_ACCESS_KEY_ID", "minio")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "minio123")
os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")
ENDPOINT = "http://localhost:9000"; BUCKET = "datalake-seg"

import boto3
from botocore.client import Config
s3 = boto3.client("s3", endpoint_url=ENDPOINT, config=Config(signature_version="s3v4"))
try:
    s3.create_bucket(Bucket=BUCKET)
except Exception:
    pass
# un "modelo" de ejemplo para subir
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_breast_cancer
d = load_breast_cancer(); modelo = LogisticRegression(max_iter=5000).fit(d.data, d.target)
print("Bucket listo:", BUCKET)

## 1. Integridad — subir el modelo con checksum y verificarlo al cargar

In [ ]:
buf = io.BytesIO(); joblib.dump(modelo, buf); blob = buf.getvalue()
sha = hashlib.sha256(blob).hexdigest()
s3.put_object(Bucket=BUCKET, Key="models/v1/modelo.pkl", Body=blob)
s3.put_object(Bucket=BUCKET, Key="models/v1/modelo.sha256", Body=sha.encode())

def cargar_verificando():
    blob = s3.get_object(Bucket=BUCKET, Key="models/v1/modelo.pkl")["Body"].read()
    esperado = s3.get_object(Bucket=BUCKET, Key="models/v1/modelo.sha256")["Body"].read().decode()
    if hashlib.sha256(blob).hexdigest() != esperado:
        raise ValueError("INTEGRIDAD ROTA: el artefacto fue manipulado")
    return joblib.load(io.BytesIO(blob))

m = cargar_verificando()
print("Modelo cargado y verificado por checksum. SHA:", sha[:20], "...")

## 2. Control principal (GUIADO) — URL prefirmada con expiración (en vez de bucket público)

In [ ]:
# En vez de exponer el bucket, se entrega un enlace temporal y acotado:
url = s3.generate_presigned_url("get_object",
        Params={"Bucket": BUCKET, "Key": "models/v1/modelo.pkl"},
        ExpiresIn=60)   # válido 60 segundos
print("URL prefirmada (expira en 60 s):")
print(url[:90], "...")

# Política de acceso MÍNIMO (ejemplo conceptual): solo lectura de models/ para un rol.
politica = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow", "Action": ["s3:GetObject"],
        "Resource": f"arn:aws:s3:::{BUCKET}/models/*"   # nada más que models/
    }]
}
print("\nPolítica de acceso mínimo (solo lectura de models/):")
print(json.dumps(politica, indent=2))

## 3. Tu entrega — completa los `# TODO`

In [ ]:
# TODO 1: subí el artefacto de TU modelo con su checksum y verifica la carga.
# TODO 2: implementa un 'registro de acceso' (audit log): cada get/put a una
#   lista o archivo con (ts, operación, key). Es tu medida de gobernanza.
# TODO 3: separa zonas con permisos distintos (p.ej. raw/ solo escritura del
#   ingestor, curated/ solo lectura del serving) y documenta la política.
print("Completa los TODO sobre tu propio modelo en el lake.")

## 4. Reflexión
1. ¿Por qué una URL prefirmada con expiración es más segura que un bucket público?
2. ¿Qué ataque detecta el checksum del modelo?
3. ¿Qué elemento de SAIF cubre tu control principal?

*(Escribe tu respuesta aquí.)*